# 02 — Checking the Gaussian Emission Assumption

Program 1's baseline HMM (`src/regime_detection.py`) assumes each regime's daily returns are drawn from a Gaussian distribution. This notebook checks whether that assumption actually holds by taking the model's own decoded regime for each day and testing the returns *within* each detected regime for normality: skewness, excess kurtosis, the Jarque-Bera test, and QQ-plots against a normal distribution.

This depends on `notebooks/01_synthetic_data.ipynb` having been run first (to produce `data/synthetic_nq.csv`).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

%matplotlib inline

from src.regime_detection import fit_and_label

DATA_PATH = PROJECT_ROOT / "data" / "synthetic_nq.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "data/synthetic_nq.csv not found — run notebooks/01_synthetic_data.ipynb first."
    )

df = pd.read_csv(DATA_PATH, parse_dates=["date"], index_col="date")
returns = df["log_return"].values

## Fit the model and take its own decoded regimes

We deliberately use `detected_regime` (the HMM's own Viterbi-decoded state, mapped to a label by `label_states`) rather than the synthetic data's ground-truth `regime` column. The question this notebook asks is whether the model's *own* Gaussian assumption is consistent with what it actually classifies as each regime — not whether it recovered the true regimes (that was already checked in `tests/test_regime_detection.py`, where it did not do so reliably on this fat-tailed data).

In [ ]:
model, state_labels, predicted = fit_and_label(returns)
df["detected_regime"] = predicted

df["detected_regime"].value_counts()

## Metrics

For each detected regime's returns we compute:

- **Skewness** — 0 for a symmetric distribution like the normal; equity-index returns are often left-skewed (sharp selloffs, slower rallies).
- **Excess kurtosis** — 0 for a normal distribution; positive values mean fatter tails and more extreme outliers than a Gaussian predicts.
- **Jarque-Bera test** — a joint test of sample skewness and kurtosis against the normal distribution's values (0 and 0). Null hypothesis: the data is normally distributed. We reject normality at p < 0.05.

In [ ]:
REGIME_ORDER = ["bull", "bear", "sideways"]
regime_returns = {r: df.loc[df["detected_regime"] == r, "log_return"].values for r in REGIME_ORDER}

rows = []
for regime in REGIME_ORDER:
    r = regime_returns[regime]
    jb_stat, jb_p = stats.jarque_bera(r)
    rows.append({
        "regime": regime,
        "n": len(r),
        "mean": r.mean(),
        "std": r.std(),
        "skewness": stats.skew(r),
        "excess_kurtosis": stats.kurtosis(r),
        "jarque_bera_stat": jb_stat,
        "jarque_bera_p": jb_p,
        "rejects_normality (p<0.05)": jb_p < 0.05,
    })

summary = pd.DataFrame(rows).set_index("regime")
summary

## QQ-plots against a normal distribution

A QQ-plot sorts the sample and compares it to the theoretical quantiles of a normal distribution. If the data were Gaussian, points would sit on the diagonal reference line; systematic curvature away from the line at the ends indicates fatter (or thinner) tails than normal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, regime in zip(axes, REGIME_ORDER):
    stats.probplot(regime_returns[regime], dist="norm", plot=ax)
    ax.set_title(f"{regime.capitalize()} (n={len(regime_returns[regime])})")
plt.tight_layout()
plt.show()

## Saving report artifacts

The combined grid above is for exploring the notebook interactively. Save
each regime's QQ-plot as its own PNG, and the Jarque-Bera results as plain
text, under `reports/` — a plain output folder for artifacts a reviewer (or
another tool) might want without re-running the notebook.

In [ ]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

for regime in REGIME_ORDER:
    fig, ax = plt.subplots(figsize=(6, 5))
    stats.probplot(regime_returns[regime], dist="norm", plot=ax)
    ax.set_title(f"{regime.capitalize()} (n={len(regime_returns[regime])})")
    fig.tight_layout()
    out_path = REPORTS_DIR / f"qq_plot_{regime}.png"
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"Saved {out_path}")

In [ ]:
lines = ["Jarque-Bera Normality Test Results (detected regimes)", "=" * 55]
for regime in REGIME_ORDER:
    n = summary.loc[regime, "n"]
    jb_stat = summary.loc[regime, "jarque_bera_stat"]
    jb_p = summary.loc[regime, "jarque_bera_p"]
    lines.append(f"{regime.upper():<10} n={n:<6} JB statistic={jb_stat:>10.4f}   p-value={jb_p:.6e}")

report_text = "\n".join(lines)
print(report_text)

report_path = REPORTS_DIR / "jarque_bera_results.txt"
report_path.write_text(report_text + "\n")
print(f"\nSaved {report_path}")

## For context: the same test against the true (ground-truth) regimes

The detected `bear` state above has only a handful of days in it — the HMM's decoded "bear" is really just chasing the most extreme outlier returns rather than capturing the full regime (consistent with `tests/test_regime_detection.py`, which found this Gaussian baseline recovers the true regimes with only middling accuracy on fat-tailed data). That makes the detected-regime test underpowered for that state. As a sanity check, here's the identical test run against the synthetic data's true `regime` labels instead, so the small detected-bear sample doesn't get mistaken for evidence that bear-market returns are actually Gaussian.

In [ ]:
true_rows = []
for regime in REGIME_ORDER:
    r = df.loc[df["regime"] == regime, "log_return"].values
    jb_stat, jb_p = stats.jarque_bera(r)
    true_rows.append({
        "regime": regime,
        "n": len(r),
        "skewness": stats.skew(r),
        "excess_kurtosis": stats.kurtosis(r),
        "jarque_bera_p": jb_p,
        "rejects_normality (p<0.05)": jb_p < 0.05,
    })

pd.DataFrame(true_rows).set_index("regime")

## Summary: does the data support the Gaussian assumption?

**No — the Gaussian emission assumption is clearly violated, and the detected-regime results actually understate how badly.**

- Against the model's own **detected** regimes, results were mixed on the surface: `sideways` (1,007 of 1,260 days) strongly rejects normality (Jarque-Bera p ≈ 3×10⁻⁷, excess kurtosis ≈ 0.79), while `bull` (247 days) and `bear` (6 days) technically fail to reject it. But that's not evidence those regimes *are* Gaussian — it's a symptom of two other problems: the decoded `bear` state has only 6 days in it (nowhere near enough data for the test to have any power), and `bull` absorbed a mix of days with only mildly fat tails, just not enough to clear statistical significance at this sample size. The QQ-plots make this visible even where the test didn't formally reject: `bull` and especially `sideways` both show the classic S-curve of points bowing away from the reference line in both tails.
- Against the **true, ground-truth** regime labels, the picture is unambiguous: all three regimes reject normality decisively (p-values from 10⁻⁸ to 10⁻⁸⁷), with meaningful excess kurtosis in every regime (roughly 1.9 to 4.0). This lines up with how the data was built in `01_synthetic_data.ipynb` — Student-t shocks with 5 degrees of freedom were used specifically because real NQ returns have fatter tails than a normal distribution, and that property survives being split out by regime.
- Put together: the fat tails aren't an artifact of imperfect regime detection — they're really there in every regime, and imperfect detection (a razor-thin "bear" state, a bloated "sideways" catch-all) just makes the within-regime test noisier for some states. This is direct empirical support for `ROADMAP.md`'s Phase 1 plan to move from this Gaussian baseline to a more robust skew-t emission model, which should both fit the fat tails more honestly and, plausibly, improve regime *detection* itself by giving the model an emission distribution that actually matches the data.